In [4]:
# RIT Opportunity Wise Dataset — Data Cleaning | Week 1
import pandas as pd
import numpy as np
import re

# ── Step 0: Load ──────────────────────────────────────────────────────────────
df = pd.read_csv(r"C:\Users\Hp\Downloads\RIT+Opportunity+Wise+Data+-+Sheet1.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# ── Step 1: Remove Duplicates ─────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Duplicates removed: {before - len(df)}")

# ── Step 2: Null Malformed Timestamps ─────────────────────────────────────────
def is_valid_datetime(s):
    m = re.match(r'^(\d{2})/(\d{2})/(\d{4}) (\d{2}):(\d{2}):(\d{2})$', str(s).strip())
    return bool(m) and 1 <= int(m.group(1)) <= 12 and 0 <= int(m.group(4)) <= 23

for col in ['Apply Date', 'Learner SignUp DateTime', 'Opportunity End Date']:
    bad = df[col].apply(lambda x: not is_valid_datetime(x) if pd.notna(x) else False)
    print(f"  '{col}': {bad.sum()} malformed → nulled")
    df.loc[bad, col] = np.nan

# ── Step 3: Parse Date Columns ────────────────────────────────────────────────
for col in ['Learner SignUp DateTime', 'Opportunity End Date',
            'Entry created at', 'Apply Date', 'Opportunity Start Date']:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y %H:%M:%S', errors='coerce')

df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], format='%m/%d/%Y', errors='coerce')

# ── Step 4: Standardise Institution Name ──────────────────────────────────────
before_u = df['Institution Name'].nunique()
df['Institution Name'] = df['Institution Name'].str.strip().str.title()
print(f"Institution variants collapsed: {before_u} → {df['Institution Name'].nunique()}")

# ── Step 5: Strip Whitespace from Text Columns ────────────────────────────────
text_cols = ['Opportunity Name', 'Opportunity Category', 'Opportunity Id',
             'First Name', 'Gender', 'Country', 'Current/Intended Major', 'Status Description']
df[text_cols] = df[text_cols].apply(lambda c: c.str.strip())

# ── Step 6: Anonymise Student IDs in First Name ───────────────────────────────
id_pattern = r'^(RP\d+|\d{2}-[A-Z]-\d{4}|[A-Z0-9]+-[a-z]+-\d+)'
mask = df['First Name'].str.match(id_pattern, na=False)
print(f"Student IDs in First Name: {mask.sum()} → replaced with 'Anonymized/ID'")
df.loc[mask, 'First Name'] = 'Anonymized/ID'

# ── Step 7: Document Missing Values ──────────────────────────────────────────
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(2)
print("\nMissing value summary:")
print(pd.DataFrame({'Count': missing, '%': pct})[missing > 0].to_string())
print("\nMissing Start Date by category:")
print(df[df['Opportunity Start Date'].isnull()]['Opportunity Category'].value_counts().to_string())

# ── Step 8: Validate Status Code Consistency ──────────────────────────────────
inconsistent = df.groupby('Status Code')['Status Description'].nunique()
inconsistent = inconsistent[inconsistent > 1]
print(f"\nStatus Code inconsistencies: {len(inconsistent)} (0 = fully consistent)")

# ── Step 9: Derive Age at Sign-Up ─────────────────────────────────────────────
df['Age_at_Signup'] = ((df['Learner SignUp DateTime'] - df['Date of Birth']).dt.days / 365.25).round(1)
df.loc[(df['Age_at_Signup'] < 10) | (df['Age_at_Signup'] > 80), 'Age_at_Signup'] = np.nan
print(f"\nAge_at_Signup — Mean: {df['Age_at_Signup'].mean():.1f} | "
      f"Min: {df['Age_at_Signup'].min()} | Max: {df['Age_at_Signup'].max()}")


Loaded: 8558 rows, 16 columns
Duplicates removed: 0
  'Apply Date': 307 malformed → nulled
  'Learner SignUp DateTime': 295 malformed → nulled
  'Opportunity End Date': 1262 malformed → nulled
Institution variants collapsed: 2089 → 1818
Student IDs in First Name: 2 → replaced with 'Anonymized/ID'

Missing value summary:
                         Count      %
Learner SignUp DateTime    295   3.45
Opportunity End Date      1262  14.75
Institution Name             5   0.06
Current/Intended Major       5   0.06
Apply Date                 307   3.59
Opportunity Start Date    4637  54.18

Missing Start Date by category:
Opportunity Category
Internship     4433
Course          116
Competition      66
Event            18
Engagement        4

Status Code inconsistencies: 0 (0 = fully consistent)

Age_at_Signup — Mean: 24.0 | Min: 13.0 | Max: 57.5
